In [1]:
from __future__ import print_function
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

# Hyperbolic Equations - Part I

Persamaan hiperbolik sudah sangat dikenal dan muncul dalam banyak model sistem fisik serta model lain dengan perilaku seperti gelombang. Salah satu persamaan diferensial parsial hiperbolik yang paling mendasar adalah persamaan adveksi.
$$
    u_t + a u_x = 0
$$
di mana $u(x,t)$ yang tidak diketahui adalah suatu besaran yang diangkut oleh aliran dengan kecepatan $a$. Persamaan ini akan menjadi dasar pembahasan kita mengenai topik yang luas ini.

Untuk masalah Cauchy (domain spasial tak terbatas) kita hanya membutuhkan kondisi awal $u(x,0) = u_0(x)$ yang dengannya kita dapat langsung menuliskan solusi dari PDE sederhana ini sebagai
$$
    u(x,t) = u_0(x - a t).
$$
Kesederhanaan ini menyembunyikan kompleksitas ketika menyangkut metode numerik untuk menyelesaikan persamaan ini. Untungnya, persamaan ini menunjukkan banyak kesulitan numerik dalam menyelesaikan PDE hiperbolik yang lebih kompleks, sementara pada saat yang sama sangat mudah untuk diselesaikan.

Jika kita memiliki batas-batas terbatas pada persamaan adveksi, perhatikan bahwa untuk konsistensi kita hanya membutuhkan satu batas, yang disebut batas aliran masuk, dan tidak ada batas aliran keluar. Batas mana yang mana ditentukan oleh arah aliran, yaitu tanda $a$. Secara numerik ini dapat menyebabkan masalah tergantung pada diskretisasi yang kita gunakan (misalnya, jika kita membutuhkan titik dari batas aliran keluar). Sebaliknya, dalam praktiknya kita akan mengatasi hal ini dengan menetapkan batas secara numerik yang tidak bertentangan dengan solusi aliran keluar atau dengan menggunakan aproksimasi satu sisi yang berbeda. Kita akan kembali membahas masalah ini sedikit kemudian, tetapi ada baiknya untuk menyadari hal ini saat kita membahasnya.

## First Discretizations

Pendekatan pertama mungkin adalah mendiskretisasi persamaan ini dengan metode Euler maju dalam waktu dan perbedaan terpusat orde kedua dalam ruang yang mengarah ke diskretisasi.
$$
    \frac{U^{n+1}_j - U^n_j}{\Delta t} = \frac{a}{2 \Delta x} (U^n_{j+1} - U^n_{j-1})
$$
or 
$$
    U^{n+1}_j = U^n_j + \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1}).
$$
Ternyata metode ini tidak terlalu berguna dalam praktiknya, seperti yang akan kita lihat saat mempertimbangkan stabilitas.

Diskretisasi umum lain dari persamaan adveksi adalah yang ditemukan dari Leapfrog (titik tengah). Dengan menggunakan pendekatan ini, kita menemukan
$$
    \frac{U^{n+1}_j - U^{n-1}_j}{2 \Delta t} + a \frac{U^n_{j+1} - U^n_{j-1}}{2 \Delta x} = 0
$$
or in update form
$$
    U^{n+1}_j = U^{n-1}_j - \frac{a \Delta t }{\Delta x} (U^n_{j+1} - U^n_{j-1}).
$$

Sebagai catatan terakhir sebelum beralih ke analisis konvergensi, kita akan membahas hubungan antara $\Delta t$ dan $\Delta x$. Misalnya, untuk metode Lax-Friedrichs, kita akan menemukan hubungan tersebut.
$$
    \left | \frac{a \Delta t}{\Delta x} \right | \leq 1
$$
yang, selain $a$, menunjukkan bahwa $\Delta t$ dan $\Delta x$ dapat berubah pada orde yang sama! Ini membawa kita pada kesimpulan umum bahwa PDE hiperbolik pada umumnya kurang kaku dibandingkan PDE parabolik karena turunan yang terlibat. Inilah juga mengapa PDE hiperbolik dapat diselesaikan secara efektif menggunakan diskretisasi langkah waktu eksplisit daripada diskretisasi implisit yang kita anggap berguna untuk PDE parabolik.

## Metode Diskretisasi Garis
Sebagai langkah awal untuk memahami stabilitas diskretisasi yang telah kita sebutkan di atas, mari kita rumuskan persamaan adveksi dengan pendekatan metode garis dan analisis stabilitasnya dalam hal masalah nilai awal.

Untuk mempertimbangkan semua metode sebelumnya, kita akan mengasumsikan kondisi batas periodik (agar tidak memiliki masalah aliran masuk/keluar). Kondisi batas ini terlihat seperti
$$
    u(0, t) = u(1, t), ~~~ \text{for}~ t \geq 0
$$
di mana kami memilih $\Omega = [0, 1]$ untuk mempermudah. ​​Pengaturan ini juga mirip dengan masalah Cauchy asli.

Sekarang kita memperkenalkan vektor yang tidak diketahui.
$$
    U = \begin{bmatrix}
        U_1(t) \\ U_2(t) \\ \vdots \\ U_m(t) \\ U_{m+1}(t)
    \end{bmatrix}
$$
where we have included an extra value at $x_{m+1}$ with the idea that $U_0(t) = U_{m+1}(t)$.  Using this discretization we have the system of ODEs

$$
    U'_j(t) = -\frac{a}{2 \Delta x} \left \{ \begin{aligned}
        &(U_2(t) - U_{m+1}(t)) & & j = 1 \\
        &(U_{j+1}(t) - U_{j-1}(t)) & & 2 \leq j \leq m \\
        &(U_1(t) - U_{m}(t)) & & j = m + 1
    \end{aligned} \right .
$$

Hal ini mengarah pada sistem $U'(t) = A U(t)$ dengan
$$
    A = -\frac{a}{2 \Delta x} \begin{bmatrix}
        0  &  1 &  & & & -1\\
        -1 &  0 & 1 \\
           & -1 & 0 & 1 \\
         & & \ddots & \ddots & \ddots \\
         & & & -1 & 0 & 1 \\
         1 & & & & -1 & 0 \\
    \end{bmatrix}.
$$
Nilai-nilai di sudut-sudut tersebut menunjukkan banyak persamaan diferensial parsial (PDE) dengan kondisi batas periodik.

Matriks $A$ ternyata *simetris miring* ($A^T = - A$) yang berarti nilai eigennya murni imajiner. Nilai eigen ini adalah
$$
    \lambda_p = -\frac{i a}{\Delta x} \sin(2 \pi p \Delta x) \quad \text{for } p = 1, 2, \ldots , m+1
$$
dengan vektor eigen yang bersesuaian
$$
    u^p_j = e^{2 \pi i p j / \Delta x} \quad \text{for} ~ j = 1, 2, \ldots m + 1.
$$

Mengingat bahwa semua nilai eigen terletak di sepanjang sumbu imajiner
$$
    \lambda \in \left[ -\frac{i a}{\Delta x}, \frac{i a}{\Delta x} \right]
$$
we conclude that with the spatial discretization above that we need a method with an absolute stability region that includes the imaginary axis
$$
    z \in \left[ -\frac{i a \Delta t}{\Delta x}, \frac{i a \Delta t}{\Delta x} \right]
$$
Sekarang juga memperhitungkan $\Delta t$. Metode yang mungkin digunakan antara lain metode titik tengah, dan beberapa metode Adams mungkin cocok, sedangkan metode BDF tidak akan cocok.

### Example:  Forward Euler
Pendekatan pertama yang kami sajikan di atas menggunakan diskretisasi Euler maju dalam waktu. Jika kita melihat wilayah stabilitas absolut, kita tahu
$$
    |1 + \Delta t \lambda| \leq 1
$$
yang merupakan lingkaran satuan yang berpusat di -1. Rasio $\Delta t / \Delta x$ tidak akan pernah menghasilkan metode yang stabil karena wilayah ini hanya mencakup titik $z = 0$ dalam skenario ini. Jika sebaliknya kita membiarkan $\Delta t$ mendekati nol lebih cepat daripada $\Delta x$, kita mungkin dapat menyebabkan $\lambda_p$ dari matriks menyusut ke titik asal.

Dalam pembahasan mengenai stabilitas Lax-Richtmyer, kita menggunakan batas yang lebih lemah di sini $||B|| \leq 1 + \alpha \Delta t$ di mana $B = I + \Delta t A$. Dengan menggunakan fakta bahwa $\lambda_p$ adalah bilangan imajiner murni dan memilih $\Delta t = \Delta x^2$, kita tahu
$$\begin{aligned}
    | 1 + \Delta t \lambda_p |^2 &\leq 1 + \left(\frac{a \Delta t}{\Delta x}\right)^2 \\
    & \leq 1 + a^2 \Delta x^2 = 1 + a^2 \Delta t.
\end{aligned}$$
We can then bound $||B||$ by
$$
    ||B|| \leq 1 + \alpha \Delta t
$$
with $\alpha = a^2$.  If $n \Delta t \leq T$ we then know
$$
    \left\|(I + \Delta t A)^n \right\|_2 \leq (1 + a^2 \Delta t)^{n / 2} \leq e^{a^2 T / 2}
$$
dan oleh karena itu metode ini stabil menurut Lax-Richtmyer.

### Example:  Lax-Friedrichs
Ada baiknya untuk menulis ulang metode Lax-Friedrichs di atas.
$$
    U^{n+1}_j = \frac{1}{2}(U^n_{j-1} + U^n_{j+1}) - \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1})
$$
menggunakan fakta bahwa
$$
    \frac{1}{2}(U^n_{j-1} + U^n_{j+1}) = U^n_j + \frac{1}{2} (U^n_{j-1} - 2 U^n_j + U^n_{j+1})
$$
sebagai
$$
    U^{n+1}_j = U^n_j + \frac{1}{2} (U^n_{j-1} - 2 U^n_j + U^n_{j+1}) - \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1}).
$$
Perhatikan kemiripan antara rata-rata tertimbang yang baru dan stensil untuk aproksimasi selisih terpusat orde kedua terhadap turunan kedua.

Hal ini dapat diatur ulang lebih lanjut untuk memberikan interpretasi diskretisasi langsung dari metode tersebut.
$$
    \frac{U^{n+1}_j - U^n_j}{\Delta t} + a \left(\frac{U^n_{j+1} - U^n_{j-1}}{2 \Delta x} \right ) = \frac{\Delta x^2}{2 \Delta t} \left( \frac{U^n_{j-1} - 2 U^n_j + U^n_{j+1}}{\Delta x^2} \right ).
$$

Memeriksa konsistensi aproksimasi di sebelah kiri mengarah pada
$$\begin{aligned}
    &\frac{1}{\Delta t} \left [ u + \Delta t u_t + \frac{\Delta t^2}{2} u_{tt} + \frac{\Delta t^3}{6} u_{ttt} + \mathcal{O}(\Delta t^4) - u \right ] \\
    & \quad \quad + \frac{a}{2 \Delta x}\left [ u + \Delta x u_x + \frac{\Delta x^2}{2} u_{xx} + \frac{\Delta x^3}{6} u_{xxx} -u + \Delta x u_x - \frac{\Delta x^2}{2} u_{xx} + \frac{\Delta x^3}{6} u_{xxx}) + \mathcal{O}(\Delta x^4) \right ] \\
    &= u_t + a u_x + \frac{\Delta t}{2} u_{tt} + \frac{\Delta t^2}{6} u_{ttt} + a \frac{\Delta x^2}{6} u_{xxx} + \mathcal{O}(\Delta t^3) + \mathcal{O}(\Delta x^3)
\end{aligned}$$
Jadi kita dapat menyimpulkan bahwa jika hanya sisi kiri saja, kita akan konsisten tetapi kita memiliki suku tambahan!

Dengan memperluas sisi kanan pada deret Taylor, kita melihat...
$$\begin{aligned}
    \frac{\Delta x^2}{2 \Delta t} \left(u_{xx} + \frac{\Delta x^2}{12} u_{xxxx} + \mathcal{O}(\Delta x^4) \right ) 
\end{aligned}$$
yang mengarah pada kesimpulan bahwa ini adalah diskretisasi dari persamaan adveksi-difusi
$$
    u_t + a u_x = \epsilon u_{xx}
$$
Di mana
$$
    \epsilon = \frac{\Delta x^2}{2 \Delta t}.
$$

Kita dapat menulis ulang sistem ini sebagai $U'(t) = A_\epsilon U(t)$ di mana
$$
    A_\epsilon = -\frac{a}{2 \Delta t} \begin{bmatrix}
        0  &  1 &  & & & -1\\
        -1 &  0 & 1 \\
           & -1 & 0 & 1 \\
         & & \ddots & \ddots & \ddots \\
         & & & -1 & 0 & 1 \\
         1 & & & & -1 & 0 \\
    \end{bmatrix} + \frac{\epsilon}{\Delta x^2} \begin{bmatrix}
        -2 & 1 & & & & 1\\
        1 & -2 & 1 \\
        & 1 & -2 & 1 \\
        & & \ddots & \ddots & \ddots \\
        & & & 1 & - 2 & 1 \\
        1 & & & & 1 & -2 
    \end{bmatrix}.
$$
Perhatikan secara khusus bahwa ini adalah matriks $A$ yang sama seperti pada diskretisasi Euler maju ketika $\epsilon = 0$. Karena suku tambahan dalam $A_\epsilon$ menyebabkan matriks tidak lagi miring simetris tetapi simetris, kita mengharapkan nilai eigen tidak lagi sepenuhnya imajiner. Suku tambahan ini dapat dilihat sebagai *regularisasi* atau *relaksasi* dari masalah asli melalui suku difusif. Penting juga untuk dicatat bahwa banyak diskretisasi praktis dari persamaan adveksi seringkali bersifat difusif, baik sebagai konsekuensi dari diskretisasi atau sebagai fitur bawaan seperti yang kita lihat di sini.

In [ ]:
# Implementasikan Lax-Friedrichs untuk PDE u_t + u_x = 0 pada domain periodik
#domain
u_true = lambda x, t: numpy.exp(-20.0 * ((x - t) - 2.0)**2) + numpy.exp(-((x - t) - 5.0)**2)

m = 501
x = numpy.linspace(0, 25.0, m)
delta_x = 25.0 / (m - 1)
cfl = 0.8
delta_t = cfl * delta_x

U = u_true(x, 0)
U_new = numpy.empty(U.shape)
t = 0.0
t_final = 17.0
while t < t_final:
    U_new[0] = 0.5 * (U[1] + U[-1]) - delta_t / (2.0 * delta_x) * (U[1] - U[-1])
    U_new[1:-1] = 0.5 * (U[2:] + U[:-2]) - delta_t / (2.0 * delta_x) * (U[2:] - U[:-2])
    U_new[-1] = 0.5 * (U[0] + U[-2]) - delta_t / (2.0 * delta_x) * (U[0] - U[-2])
    U = U_new.copy()
    t += delta_t

# Plot solusi pada t = 17.0 dan t = 0.0
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'ro')
axes.plot(x, u_true(x, t),'k')
axes.set_xlim((15.0, 25.0))
axes.set_ylim((-0.1, 1.1))

plt.show()

In [ ]:
# Plot eigenvalues of the matrix A_\epsilon and plot relative 
# absolute stability region

def construct_A(epsilon, a=1.0, delta_x=0.02):
    """Construct the matrix A from Leveque (10.15)
    """
    e = numpy.ones(int(1.0 / delta_x))
    A = numpy.diag(e[1:], 1) - numpy.diag(e[1:], -1)
    A[0, -1] = -1
    A[-1, 0] = 1
    
    B = numpy.diag(-2.0 * e, 0) + numpy.diag(e[1:], 1) + numpy.diag(e[1:], -1)
    B[0, -1] = 1
    B[-1, 0] = 1
    
    return -a / (2.0 * delta_x) * A + epsilon / delta_x**2 * B
    
delta_x = 0.02
delta_t = 0.8 * delta_x
a = 1.0
fig = plt.figure()
fig.set_figwidth(fig.get_figwidth() * 2)
fig.set_figheight(fig.get_figheight() * 4)
titles = ["Forward Euler", "", "", "Lax-Wendroff", "Lax-Friedrichs", ""]
for (i, epsilon) in enumerate((0.0, 0.001, 0.005, 0.008, 0.0125, 0.014)):
    axes = fig.add_subplot(4, 2, i + 1, aspect='equal')
    
    # Plot eigenvalues
    eigenvalues = numpy.linalg.eigvals(construct_A(epsilon, a, delta_x))
    axes.plot(delta_t * eigenvalues.real, delta_t * eigenvalues.imag, 'ro')
    
    # Plot offset circle
    theta = numpy.linspace(0.0, 2.0 * numpy.pi, 100)
    axes.plot(numpy.sin(theta) - 1.0, numpy.cos(theta), 'k')
    
    axes.set_xlim((-2.5, 0.5))
    axes.set_ylim((-1.5, 1.5))
    axes.set_title("$\epsilon$ = %s, %s" % (epsilon, titles[i]))
    
    axes.plot([-2.5, 0.5], [0.0, 0.0], color='lightgray')
    axes.plot([0.0, 0.0], [-1.5, 1.5], color='lightgray')
    
plt.show()

### Contoh: Lompatan Katak

Untuk metode lompatan katak (titik tengah)
$$
    U^{n+1}_j = U^{n-1}_j - \frac{a \Delta t}{\Delta x} (U^n_{j+1} - U^n_{j-1})
$$
Kita tahu bahwa daerah stabilitasnya adalah interval sepanjang sumbu imajiner $z \in [-i, i]$. Ini menyiratkan melalui metode garis bahwa jika
$$
    \left | \frac{a \Delta t}{\Delta x} \right | \leq 1
$$
Metode ini akan stabil. Satu-satunya pengecualian adalah bahwa $z = \Delta t \lambda_p$ akan selalu berada di batas wilayah ini (semuanya demikian), yang berarti jika kita melihat sedikit gangguan pada nilai eigen, kita mungkin akan mengalami masalah. Tidak ada pertumbuhan tetapi juga tidak ada peluruhan dari mode eigen apa pun dari aproksimasi, yang juga dikenal sebagai *nondisipatif*. Ini bukan perilaku yang buruk, persamaan adveksi itu sendiri bersifat nondisipatif, tetapi masalah lainnya adalah bahwa mode bergerak dengan kecepatan yang berbeda daripada yang ditentukan oleh $a$, yang menyebabkan kesalahan dispersif. Ini juga bisa menjadi sulit jika masalahnya lebih kompleks daripada persamaan adveksi (misalnya, masalah tidak homogen atau nonlinier).


In [ ]:
# Implementasikan Leapfrog untuk PDE u_t + u_x = 0 pada sistem periodik
# domain
u_true = lambda x, t: numpy.exp(-20.0 * ((x - t) - 2.0)**2) + numpy.exp(-((x - t) - 5.0)**2)

m = 501
x = numpy.linspace(0, 25.0, m)
delta_x = 25.0 / (m - 1)
cfl = 0.8
delta_t = cfl * delta_x

U = u_true(x, 0)
t = 0.0
t_final = 17.0
# Jump start with Lax-Friedrichs
U_new = numpy.empty(U.shape)
U_new[0] = 0.5 * (U[1] + U[-1]) - delta_t / (2.0 * delta_x) * (U[1] - U[-1])
U_new[1:-1] = 0.5 * (U[2:] + U[:-2]) - delta_t / (2.0 * delta_x) * (U[2:] - U[:-2])
U_new[-1] = 0.5 * (U[0] + U[-2]) - delta_t / (2.0 * delta_x) * (U[0] - U[-2])
U_old = U_new.copy()
t += delta_t
while t < t_final:
    U_new[0] = U_old[0] - delta_t / delta_x * (U[1] - U[-1])
    U_new[1:-1] = U_old[1:-1] - delta_t / delta_x * (U[2:] - U[:-2])
    U_new[-1] = U_old[-1] - delta_t / delta_x * (U[0] - U[-2])
    U_old = U.copy()
    U = U_new.copy()
    t += delta_t
    
# Plot solution at t = 17.0 and t = 0.0
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'ro')
axes.plot(x, u_true(x, t),'k')
axes.set_xlim((15.0, 25.0))
axes.set_ylim((-0.1, 1.1))

plt.show()

## Metode Lax-Wendroff
Sejauh ini satu-satunya metode yang telah kita gunakan yang merupakan orde kedua dalam ruang **dan** waktu adalah diskretisasi leapfrog, tetapi kita melihat bahwa kita memiliki masalah dengan perilaku nondisipatifnya sekaligus merupakan metode multi-langkah. Mari kita jelajahi pendekatan lain yang masih merupakan metode satu langkah. Pertimbangkan opsi berikut:

1. Jadikan metode tersebut implisit (aturan trapesium)
2. Gunakan metode Runge-Kutta (RK 2)
3. Gunakan metode deret Taylor

1. Jadikan metode implisit (aturan trapesium) - Kita telah menyebutkan bahwa kita seharusnya tidak perlu menggunakan metode implisit karena persamaan adveksi (dan banyak PDE hiperbolik) tidak kaku.
2. Gunakan metode Runge-Kutta (RK 2) - Ini menjadi kompleks di dekat batas dan mungkin memerlukan penyimpanan tambahan seperti halnya dengan leapfrog.
3. Gunakan metode deret Taylor - Ini mungkin memerlukan beberapa evaluasi diskretisasi spasial tetapi mengarah ke apa yang disebut *metode Lax-Wendroff*.

Luangkan sedikit waktu dan lihat apakah Anda dapat menurunkan metode Lax-Wendroff dengan melakukan ekspansi dalam turunan waktu dan menggunakan diskretisasi spasial orde kedua melalui pendekatan metode garis.

Ekspansi terhadap waktu menggunakan deret Taylor mengarah pada sistem persamaan diferensial biasa (ODE).
$$
    U^{n+1} = U^n + \Delta t A U^n + \frac{\Delta t^2}{2} A^2 U^n + \mathcal{O}(\Delta t^3)
$$
di mana $A$ adalah matriks yang diperoleh dari aproksimasi selisih terpusat orde kedua asli kita.

Dengan menuliskannya, kita mendapatkan rumus pembaruan.
$$
    U^{n+1}_j = U^n_j + \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1} ) + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U^n_{j-2} - 2 U^n_{j} + U^n_{j+2}).
$$
Perhatikan bahwa stensil sekarang menjadi lebih lebar karena istilah $A^2$ dan mungkin menimbulkan masalah dengan batas-batasnya.

Sebaliknya, kita dapat mengamati bahwa suku terakhir tampak seperti perkiraan untuk $u_{xx}$ dan alih-alih menggunakan stensil yang lebih lebar, kita dapat menggunakan stensil yang lebih familiar. Alih-alih melakukan hal di atas yang sesuai dengan pendekatan metode garis, kita dapat menggunakan ekspansi Taylor pada PDE asli dan cukup mempertahankan lebih banyak suku (mirip dengan metode ekspansi Taylor yang dibahas sebelumnya) sehingga kita dapat menurunkan metode yang lebih sederhana.

Ekspansi yang relevan adalah
$$
    u(x, t + \Delta t) = u(x,t) + \Delta t u_t(x,t) + \frac{\Delta t^2}{2} u_{tt}(x,t) + \frac{\Delta t^3}{6} u_{ttt}(x,t) + \mathcal{O}(\Delta t^4)
$$
dan dengan asumsi kelancaran yang dibutuhkan, kita dapat mengganti turunan $t$ dengan turunan $x$ dan persamaan aslinya yang kita temukan
$$
    u(x, t + \Delta t) = u(x,t) - a \Delta t u_x(x,t) + \frac{a^2 \Delta t^2}{2} u_{xx}(x,t) + \frac{a^3 \Delta t^3}{6} u_{xxx}(x,t) + \mathcal{O}(\Delta t^4)
$$

Kita melihat bahwa kita memiliki metode yang memiliki perkiraan sebagai berikut
$$\begin{aligned}
    U^{n+1}_j &= U^n_j - a \Delta t D_0 U^n_j + \frac{a^2 \Delta t^2}{2} D_2 U^n_j + \frac{a^3 \Delta t^3}{6} u_{xxx}(x,t) + \mathcal{O}(\Delta t^4) \\
    &= U^n_j - \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1})  + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U^n_{j+1} - 2 U^n_{j} + U^n_{j-1}) + \frac{a^3 \Delta t^3}{6} u_{xxx}(x,t) + \mathcal{O}(\Delta t^4)
\end{aligned}$$
mengarah ke metode
$$
     U^{n+1}_j = U^n_j - \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1})  + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U^n_{j+1} - 2 U^n_{j} + U^n_{j-1}).
$$
Dari sini kita melihat bahwa kita memang dapat menggunakan stensil kecil dan bahwa kesalahan dominan bersifat dispersif (turunan x orde ketiga).

In [ ]:
# Implementasikan Lax-Wendroff untuk PDE u_t + u_x = 0 pada sistem periodik
# domain
u_true = lambda x, t: numpy.exp(-20.0 * ((x - t) - 2.0)**2) + numpy.exp(-((x - t) - 5.0)**2)

m = 501
x = numpy.linspace(0, 25.0, m)
delta_x = 25.0 / (m - 1)
cfl = 0.8
delta_t = cfl * delta_x

U = u_true(x, 0)
U_new = numpy.empty(U.shape)
t = 0.0
t_final = 17.0
while t < t_final:
    U_new[0] = U[0] - delta_t / (2.0 * delta_x) * (U[1] - U[-1]) \
                    + delta_t**2 / (2.0 * delta_x**2) * (U[1] - 2.0 * U[0] + U[-1])
    U_new[1:-1] = U[1:-1] - delta_t / (2.0 * delta_x) * (U[2:] - U[:-2]) \
                          + delta_t**2 / (2.0 * delta_x**2) * (U[2:] - 2.0 * U[1:-1] + U[:-2])
    U_new[-1] = U[-1] - delta_t / (2.0 * delta_x) * (U[0] - U[-2]) \
                      + delta_t**2 / (2.0 * delta_x**2) * (U[0] - 2.0 * U[-1] + U[-2])
    U = U_new.copy()
    t += delta_t
    
# Plot solution at t = 17.0 and t = 0.0
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'ro')
axes.plot(x, u_true(x, t),'k')
axes.set_xlim((15.0, 25.0))
axes.set_ylim((-0.3, 1.1))

plt.show()

### Stability
Analisis metode Lax-Wendroff mengikuti pendekatan yang kita ambil untuk metode Lax-Friedrichs dengan mempertimbangkan metode umum menggunakan matriks $A_\epsilon$. Namun di sini, alih-alih $\epsilon = \Delta x^2 / 2 \Delta t$, kita menggunakan $\epsilon = a^2 \Delta t / 2$. Nilai eigen dari $A_\epsilon$ ini adalah:
$$
\Delta t \lambda_p = -i \frac{a \Delta t}{\Delta x} \sin(p \pi \Delta x) + \left( \frac{a \Delta t}{\Delta x} \right)^2 (\cos(p \pi \Delta x) - 1).



Dari contoh numerik di atas, kita melihat bahwa ini berada di dalam metode Euler dan memiliki batasan stabilitas yang sama dengan Lax-Friedrichs. Ini bagus karena kita mengharapkan Lax-Wendroff memiliki akurasi orde kedua dalam ruang dan waktu, sedangkan Lax-Friedrichs hanya orde pertama.

Pengamatan penting lainnya adalah bahwa nilai eigen untuk metode Lax-Wendroff tampaknya "optimal" mendekati batas wilayah stabilitas. Hal ini disebabkan oleh sifat orde kedua dari metode Lax-Wendroff dan memiliki sifat yang baik yaitu menggunakan jumlah redaman minimal yang dibutuhkan untuk tetap stabil.

In [ ]:
# Plot nilai eigen dari matriks A_\epsilon dan plot wilayah stabilitas relatif
# wilayah stabilitas absolut

def construct_A(epsilon, a=1.0, delta_x=0.02):
    """Construct the matrix A from Leveque (10.15)
    """
    e = numpy.ones(int(1.0 / delta_x))
    A = numpy.diag(e[1:], 1) - numpy.diag(e[1:], -1)
    A[0, -1] = -1
    A[-1, 0] = 1
    
    B = numpy.diag(-2.0 * e, 0) + numpy.diag(e[1:], 1) + numpy.diag(e[1:], -1)
    B[0, -1] = 1
    B[-1, 0] = 1
    
    return -a / (2.0 * delta_x) * A + epsilon / delta_x**2 * B
    
delta_x = 0.02
delta_t = 0.8 * delta_x
a = 1.0
fig = plt.figure()
fig.set_figwidth(fig.get_figwidth() * 2)
fig.set_figheight(fig.get_figheight() * 4)
titles = ["Forward Euler", "", "", "Lax-Wendroff", "Lax-Friedrichs", ""]
for (i, epsilon) in enumerate((0.0, 0.001, 0.005, 0.008, 0.0125, 0.014)):
    axes = fig.add_subplot(4, 2, i + 1, aspect='equal')
    
    # Plot eigenvalues
    eigenvalues = numpy.linalg.eigvals(construct_A(epsilon, a, delta_x))
    axes.plot(delta_t * eigenvalues.real, delta_t * eigenvalues.imag, 'ro')
    
    # Plot offset circle
    theta = numpy.linspace(0.0, 2.0 * numpy.pi, 100)
    axes.plot(numpy.sin(theta) - 1.0, numpy.cos(theta), 'k')
    
    axes.set_xlim((-2.5, 0.5))
    axes.set_ylim((-1.5, 1.5))
    axes.set_title("$\epsilon$ = %s, %s" % (epsilon, titles[i]))
    
    axes.plot([-2.5, 0.5], [0.0, 0.0], color='lightgray')
    axes.plot([0.0, 0.0], [-1.5, 1.5], color='lightgray')
    
plt.show()

## Metode Melawan Arah Angin
Salah satu aspek dari persamaan adveksi sederhana yang kita pertimbangkan yang belum dieksploitasi adalah asimetri dalam persamaan karena $a$. Jika $a > 0$ maka gelombang merambat ke kanan dan jika $a < 0$ gelombang merambat ke kiri. Ini menunjukkan bahwa mungkin perbedaan satu sisi sudah cukup untuk mendekati solusi daripada pendekatan terpusat yang telah kita pertimbangkan hingga saat ini.

Pertimbangkan perbedaan sepihak.
$$
    u_x(x_j, t) \approx \frac{1}{\Delta x} (U_j - U_{j-1}) \quad \text{and} \quad u_x(x_j, t) \approx \frac{1}{\Delta x} (U_{j+1} - U_{j})
$$
that are both first-order accurate approximations to the first derivative.  Using these in conjunction with a forward Euler time stepping scheme leads to
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{\Delta x} (U^n_j - U^n_{j-1}) \quad \text{or} \quad U^{n+1}_j = U^n_j - \frac{a \Delta t}{\Delta x} (U_{j+1} - U_{j})
$$
yang kemudian menjadi skema orde pertama baik dalam ruang maupun waktu.
Jadi bagaimana kita memanfaatkan asimetri yang disebutkan karena $a$ dan perbedaan sepihak kita?

Solusi sebenarnya dari persamaan adveksi pada titik $(x_j, t + \Delta t)$ adalah
$$
    u(x_j, t + \Delta t) = u(x_j - a \Delta t, t)
$$
Hal ini mewakili gagasan bahwa nilai solusi pada $u(x_j, t)$ "mengalir" ke titik baru (mengikuti karakteristik). Dalam kasus di mana $a > 0$, solusi ditentukan oleh titik di sebelah kiri $x_j$, khususnya pada $x_j - a \Delta t$. Jika $a < 0$, maka solusi ditentukan oleh titik di sebelah kanan $x_j$.

Hal ini menunjukkan bahwa kita mungkin ingin menggunakan metode tersebut.
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{\Delta x} (U^n_j - U^n_{j-1})
$$
if $a > 0$ and
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{\Delta x} (U_{j+1} - U_{j})
$$
jika $a < 0$. Metode ini disebut *metode upwind* karena menggunakan titik-titik yang berada "di hulu" dari titik saat ini untuk menemukan solusinya.

In [ ]:
# Implementasikan metode Upwind untuk PDE u_t + u_x = 0 pada sistem periodik
# domain
u_true = lambda x, t: numpy.exp(-20.0 * ((x - t) - 2.0)**2) + numpy.exp(-((x - t) - 5.0)**2)

m = 501
x = numpy.linspace(0, 25.0, m)
delta_x = 25.0 / (m - 1)
cfl = 0.8
delta_t = cfl * delta_x

U = u_true(x, 0)
U_new = numpy.empty(U.shape)
t = 0.0
t_final = 17.0
while t < t_final:
    U_new[0] = U[0] - delta_t / delta_x * (U[0] - U[-1])
    #U_new[1:-1] = U[1:-1] - delta_t / delta_x * (U[1:-1] - U[:-2])
    #U_new[-1] = U[-1] - delta_t / delta_x * (U[-1] - U[-2])
    #Periodic boundaries will wrap in vectorized operation anyways
    U_new[1:] = U[1:] - delta_t / delta_x * (U[1:] - U[:-1])
    U = U_new.copy()
    t += delta_t
    
# Plot solution at t = 17.0 and t = 0.0
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'ro')
axes.plot(x, u_true(x, t),'k')
axes.set_xlim((15.0, 25.0))
axes.set_ylim((-0.3, 1.1))

plt.show()

### Stability
Sekarang mari kita periksa stabilitas metode upwind ini. Kita dapat menulis ulang metode upwind untuk $a > 0$ dengan
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{2 \Delta x} (U^n_j - U^n_{j-1}) + \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - 2 U^n_j + U^n_{j-1})
$$
yang menghasilkan $\epsilon = a \Delta x / 2$.

Dari analisis metode garis kita, kita tahu bahwa metode seperti ini akan stabil jika...
$$
    \left |\frac{a \Delta t}{\Delta x} \right | \leq 1
$$
and
$$
    -2 < -\frac{2 \epsilon \Delta t}{\Delta x^2} < 0.
$$

Untuk metode Lax-Friedrichs, $\epsilon$ tidak bergantung pada $a$, dan untuk metode Lax-Wendroff, $\epsilon$ memiliki $a^2$ sehingga $\epsilon > 0$, dan oleh karena itu kondisi kedua ini akan terpenuhi dengan batasan yang sesuai pada $\Delta t$ dan $\Delta x$ pada batas bawah. Untuk metode upwind, hal ini tidak lagi berlaku karena tanda $a$ dapat menyebabkan $\epsilon < 0$. Bahkan, ini mengarah pada suatu kondisi yang memberi tahu kita bentuk metode upwind mana yang perlu kita gunakan tergantung pada tanda $a$.

Untuk metode yang dianalisis di atas, kita memiliki
$$
    -2 < -\frac{2 \epsilon \Delta t}{\Delta x^2} < 0 \quad \Rightarrow 0 \leq \frac{a \Delta t}{\Delta x} \leq 1
$$
yang kita ketahui akan memenuhi batas atas jika $a > 0$.

Untuk metode yang melihat ke kanan, kita memiliki kondisi berikut:
$$
    -2 < -\frac{2 \epsilon \Delta t}{\Delta x^2} < 0 \quad \Rightarrow -1 \leq \frac{a \Delta t}{\Delta x} \leq 0.
$$

Kita juga dapat kembali memplot nilai eigen matriks dan melihat apa yang terjadi dalam setiap kasus. Dengan menghubungkan nilai-nilai $\epsilon$ ini kembali ke nilai-nilai untuk Lax-Friedrichs dan Lax-Wendroff, kita peroleh
$$
    \epsilon_{LW} = \frac{a^2 \Delta t}{2} = \frac{a \Delta x \nu}{2} \quad \epsilon_{UP} = \frac{a \Delta x}{2} \quad \epsilon_{LF} = \frac{\Delta x^2}{2 \Delta t} = \frac{a \Delta x}{2 \nu}
$$
where $\nu = a \Delta t / \Delta x$ (called the Courant number).  Note that 
$$
    \epsilon_{LW} = \nu \epsilon_{UP} \quad \text{and} \quad \epsilon_UP = \nu \epsilon_{LF}
$$
dan jika $0 < \nu < 1$ maka kita tahu bahwa $\epsilon_{LW} < \epsilon_{UP} < \epsilon_{LF}$. Dengan kata lain, kita melihat lagi bahwa Lax-Wendroff dan Lax-Friedrichs membentuk batasan pada nilai $\epsilon$ yang tetap stabil dan metode upwind berada di antara kedua ekstrem tersebut.

In [ ]:
# Plot nilai eigen dari matriks A_\epsilon dan plot relatifnya
# wilayah stabilitas absolut
def construct_A(epsilon, a=1.0, delta_x=0.02):
    """Construct the matrix A from Leveque (10.15)
    """
    e = numpy.ones(int(1.0 / delta_x))
    A = numpy.diag(e[1:], 1) - numpy.diag(e[1:], -1)
    A[0, -1] = -1
    A[-1, 0] = 1
    
    B = numpy.diag(-2.0 * e, 0) + numpy.diag(e[1:], 1) + numpy.diag(e[1:], -1)
    B[0, -1] = 1
    B[-1, 0] = 1
    
    return -a / (2.0 * delta_x) * A + epsilon / delta_x**2 * B
    
delta_x = 0.02
delta_t = 0.8 * delta_x
a = 1.0
fig = plt.figure()
fig.set_figwidth(fig.get_figwidth() * 2)
titles = ["Upwind - Left", "Upwind - Right"]
for (i, epsilon) in enumerate((a * delta_x / 2.0, -a * delta_x / 2.0)):
    axes = fig.add_subplot(1, 2, i + 1, aspect='equal')
    
    # Plot eigenvalues
    eigenvalues = numpy.linalg.eigvals(construct_A(epsilon, a, delta_x))
    axes.plot(delta_t * eigenvalues.real, delta_t * eigenvalues.imag, 'ro')
    
    # Plot offset circle
    theta = numpy.linspace(0.0, 2.0 * numpy.pi, 100)
    axes.plot(numpy.sin(theta) - 1.0, numpy.cos(theta), 'k')
    
    axes.set_xlim((-2.5, 0.5))
    axes.set_ylim((-1.5, 1.5))
    axes.set_title("$\epsilon$ = %s, %s" % (epsilon, titles[i]))
    
    axes.plot([-2.5, 0.5], [0.0, 0.0], color='lightgray')
    axes.plot([0.0, 0.0], [-1.5, 1.5], color='lightgray')
    
plt.show()

### Metode Pemanasan Sinar
Anda mungkin merasa kurang nyaman karena kami kembali menggunakan metode orde pertama, tetapi jangan khawatir, *metode Pemanasan Berkas* adalah metode akurat orde kedua dengan jenis sifat satu sisi yang sama. Jika kita kembali ke ekspansi yang kita temukan saat menurunkan metode Lax-Wendroff
$$
    u(x, t + \Delta t) = u(x,t) - a \Delta t u_x(x,t) + \frac{a^2 \Delta t^2}{2} u_{xx}(x,t) + \frac{a^3 \Delta t^3}{6} u_{xxx}(x,t) + \mathcal{O}(\Delta t^4)
$$
and approximate the derivatives using one-sided differences rather than centered we get for $a > 0$
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{2 \Delta x} (3 U^n_j - 4 U^n_{j-1} + U^n_{j-2}) + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U^n_j - 2 U^n_{j-1} + U^n_{j-2})
$$
and for $a < 0$
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{2 \Delta x} (-3 U^n_j + 4 U^n_{j+1} - U^n_{j+2}) + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U^n_j - 2 U^n_{j+1} + U^n_{j+2}).
$$
Metode-metode ini stabil jika $0 \leq \nu \leq 2$ dan $-2 \leq \nu \leq 0$ secara berturut-turut.

In [ ]:
# Implementasikan metode Pemanasan Berkas untuk PDE u_t + u_x = 0 pada periodik
# domain
u_true = lambda x, t: numpy.exp(-20.0 * ((x - t) - 2.0)**2) + numpy.exp(-((x - t) - 5.0)**2)

m = 501
x = numpy.linspace(0, 25.0, m)
delta_x = 25.0 / (m - 1)
cfl = 0.8
delta_t = cfl * delta_x

U = u_true(x, 0)
U_new = numpy.empty(U.shape)
t = 0.0
t_final = 17.0
while t < t_final:
    U_new[0] = U[0] - delta_t / (2.0 * delta_x) * (3.0 * U[0] - 4.0 * U[-1] + U[-2]) \
                + delta_t**2 / (2.0 * delta_x**2) * (U[0] - 2.0 * U[-1] + U[-2])
    U_new[1] = U[1] - delta_t / (2.0 * delta_x) * (3.0 * U[1] - 4.0 * U[0] + U[-1]) \
                + delta_t**2 / (2.0 * delta_x**2) * (U[1] - 2.0 * U[0] + U[-1])
    U_new[2:] = U[2:] - delta_t / (2.0 * delta_x) * (3.0 * U[2:] - 4.0 * U[1:-1] + U[:-2]) \
                      + delta_t**2 / (2.0 * delta_x**2) * (U[2:] - 2.0 * U[1:-1] + U[:-2])
    U = U_new.copy()
    t += delta_t
    
# Plot solution at t = 17.0 and t = 0.0
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'ro')
axes.plot(x, u_true(x, t),'k')
axes.set_xlim((15.0, 25.0))
axes.set_ylim((-0.3, 1.1))

plt.show()

## Analisis Von Neumann
Sekarang kita mencoba untuk mendapatkan batasan serupa seperti yang kita lihat pada metode diskretisasi garis, tetapi menggunakan analisis von Neumann sebagai gantinya.
Ingat kembali bahwa analisis von Neumann dilakukan dengan mengganti $U^n_j$ dengan $g(\xi)^n e^{i \xi j \Delta x}$ dan mencari ekspresi untuk faktor amplifikasi $g(\xi)$. Jika faktor amplifikasi memenuhi
$$
    \left|g(\xi)\right| \leq 1
$$
then we know the method is stable.

Below we will also introduce the notation
$$
    \nu \equiv \frac{a \Delta t}{\Delta x}
$$
karena akan berguna di kemudian hari.

### Contoh - Melawan Arah Angin
Metode arah angin ke atas untuk $a > 0$ adalah
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{\Delta x}(U^n_{j} - U^n_{j - 1}).
$$
Hitung faktor amplifikasi dan kondisi stabilitas yang dihasilkan.

Dengan memasukkan solusi gelombang bidang, kita memiliki
$$\begin{aligned}
    g(\xi) &= 1 - \frac{a \Delta t}{\Delta x} (1 - e^{-i \xi \Delta x}) \\
    &= (1 - \nu) + \nu e^{-i \xi \Delta x}.
\end{aligned}$$
Faktor amplifikasi ini bervariasi dengan bilangan gelombang $\xi$ sedemikian rupa sehingga membentuk lingkaran dengan jari-jari $\nu$ yang berpusat di $1 - \nu$. Perhatikan bahwa jika $\nu = 1$, lingkaran tersebut berpusat di titik asal dan memiliki jari-jari 1, yaitu bola satuan. Dengan kata lain, mereka berada di ambang kestabilan. Ekstrem lainnya dengan $\nu = 0$ tidak begitu menarik (karena ini adalah kasus trivial dengan $a = 0$). Kesimpulannya adalah bahwa untuk stabilitas, metode upwind harus memiliki
$$
    0 \leq \nu \leq 1.
$$
Untuk kasus $a < 0$, kita menemukan kasus analog dengan $-1 \leq \nu \leq 0$.

### Contoh - Lax-Wendroff
Hitung faktor amplifikasi untuk metode Lax-Wendroff.
$$
     U^{n+1}_j = U^n_j - \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1})  + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U^n_{j+1} - 2 U^n_{j} + U^n_{j-1})
$$

$$\begin{aligned}
    g(\xi) &= 1 - \frac{\nu}{2} \left(e^{i \xi \Delta x} - e^{-i \xi \Delta x} \right ) + \frac{\nu^2}{2} \left (e^{i \xi \Delta x} - 2 + e^{-i \xi \Delta x} \right ) \\
    &= 1 - i \nu \sin(\xi \Delta x) + \nu^2 (\cos(\xi \Delta x) - 1) \\
    &= 1 - i \nu 2\sin(\xi \Delta x / 2) \cos(\xi \Delta x / 2) - \nu^2 2\sin(\xi \Delta x / 2)
\end{aligned}$$
(Baris terakhir ditulis sedemikian rupa sehingga kita dapat dengan mudah menghitung modulusnya).

$$\begin{aligned}
    |g(\xi)|^2 &= (1 - 2\nu^2 \sin^2(\xi \Delta x / 2) )^2 + 4 \nu^2 \sin^2(\xi \Delta x / 2) \cos^2(\xi \Delta x / 2) \\
    &= 1 + 4 \nu^2 \sin^2(\xi \Delta x / 2) (\cos^2(\xi \Delta x / 2) - 1) + 4\nu^4 \sin^4(\xi \Delta x / 2) \\
    &=1 -4 \nu^2 (1 - \nu^2) \sin^4(\xi \Delta x / 2).
\end{aligned}$$
Kita tahu $0 \leq \sin^4(\xi \Delta x / 2) \leq 1$ untuk setiap $\xi$, jadi kita membutuhkan $|\nu| \leq 1$ agar metode ini stabil.

### Contoh - Lax-Friedrichs
Hitung faktor amplifikasi untuk Lax-Friedrichs.
$$
    U^{n+1}_j = \frac{1}{2}(U^n_{j-1} + U^n_{j+1}) - \frac{a \Delta t}{2 \Delta x} (U^n_{j+1} - U^n_{j-1}).
$$

$$\begin{aligned}
    g(\xi) &= \frac{1}{2} (e^{i \xi \Delta x} + e^{-i \xi \Delta x}) - \frac{\nu}{2} (e^{i \xi \Delta x} - e^{-i \xi \Delta x}) \\
    &= \cos(\xi \Delta x) - i \nu \sin(\xi \Delta x)
\end{aligned}$$
leading to
$$
    |g(\xi)|^2 = \cos^2(\xi \Delta x) + \nu^2 \sin^2 (\xi \Delta x)
$$
dan oleh karena itu syaratnya adalah $|\nu| \leq 1$.

### Contoh - Lompatan Katak
Agak lebih rumit tetapi bisa dilakukan, yaitu menurunkan faktor amplifikasi dan kriteria stabilitas untuk metode leapfrog.
$$
    U^{n+1}_j = U^{n-1}_j - \frac{a \Delta t}{\Delta x} (U^n_{j+1} - U^n_{j-1})
$$

$$\begin{aligned}
     g(\xi)^2 &= 1 - \nu g(\xi) (e^{i \xi \Delta x} - e^{-i \xi \Delta x}) \\
     &= 1- 2 \nu i \sin(\xi \Delta x) g(\xi). 
\end{aligned}$$
This then implies two branches for $g(\xi)$:
$$
    g_{1,2}(\xi) =  - \nu i \sin(\xi \Delta x) \pm \sqrt{1 - \nu^2 \sin^2(\xi \Delta x)}
$$
dan oleh karena itu modulus dari
$$
    |g_{1,2}(\xi)|^2 =  \nu^2 \sin^2(\xi \Delta x) + ( 1 - \nu^2 \sin^2(\xi \Delta x)) \leq 1
$$
jika $|\nu| \leq 1$ lagi.